In [8]:
import os                 # Para acceder a variables del entorno
import uuid               # Para generar un nombre unico a los documentos de S3
import pandas as pd       # Para leer los csv y trabajar con los df
from pathlib import Path  # Para manejar el path a la carpeta con la data dentro del repo
from datetime import date # Para agregar la fecha de ingesta de los datos a la tabla

import pyarrow as pa         # Para transformar los df a formato columnar (futuro parquet)
import pyarrow.fs as pafs    # Para acceder a S3
import pyarrow.parquet as pq # Para manejar los documentos parquet

from dotenv import load_dotenv # Para agregar las variables dentro de .ven a las del entorno

In [9]:
load_dotenv()                               # Agrego las variables en .env al entorno
aws_profile = os.getenv("AWS_PROFILE")      # Tomo el perfil para conectarme a S3
bronze_lib = os.getenv("BRONZE_S3_BUCKET")  # Tomo el bucket de S3 donde voy a agregar los documentos
fs = pafs.S3FileSystem()                    # Configuro la conexion con S3

chunksize = 500_000        # Cantidad de filas que van a tener los chuncks al leer el archivo
date_col = "event_time"
ingest_date = date.today()

In [10]:
NOTEBOOK_DIR = Path().resolve()    # Carpeta del notebook actual
BASE_DIR = NOTEBOOK_DIR.parent     # Carpeta del repo 
RAW_CSV_FOLDER = BASE_DIR / "data" # Carpeta de la data raw

if not RAW_CSV_FOLDER.is_dir(): # Muestra error si no existe la carpeta
    raise NotADirectoryError(f"La carpeta de datos no existe: {RAW_CSV_FOLDER}")

file_name = input("Nombre del archivo (con extensión): ").strip() # Toma el nombre del documento
csv_path = RAW_CSV_FOLDER / file_name                             # Genero el path

if not csv_path.is_file(): # Muestra error si no existe el documento
    raise FileNotFoundError(f"No existe el archivo: {csv_path}")

In [11]:
def s3_join(*parts: str) -> str: # Evita dobles / o que falten
    return "/".join(p.strip("/") for p in parts)

In [ ]:
for chunk_number, chunk in enumerate(pd.read_csv(
    csv_path,
    chunksize=chunksize,       # Devuelve un iterador con distintos chunks para no cargar todo el df en la RAM
    parse_dates=[date_col],    # Transforma las fechas dentro del chunk y no despues todo junto
    infer_datetime_format=True
)):
    chunk = chunk.dropna(subset=[date_col])      # Quito columnas que no tengan fecha, por limpieza
    chunk["date_only"] = chunk[date_col].dt.date # Agrego una columna con solo la fecha para agrupar mas facil

    for day, daily_df in chunk.groupby("date_only"):
        
        if pd.isna(day):
            continue # Por si acaso si hay algun NaT raro

        day_folder = s3_join(bronze_lib, "events", f"event_date={day:%Y-%m-%d}") 
        parquet_path = s3_join(day_folder, f"part-{uuid.uuid4().hex}.parquet") # Genero el path para S3 incluyendo en el nombre un string de 32 caracteres para hacer el nombre unico y cuidar que hayan dos documentos con el mismo nombre.

        daily_df = daily_df.drop(columns="date_only")
        daily_df["load_date"] = ingest_date
        table = pa.Table.from_pandas(daily_df, preserve_index=False)     # Paso la tabla a formato columnar y quito el indice (columna __index_level_0__)
        
        pq.write_table(
            table,                          # Escribo la tabla columnar
            parquet_path,                   # En el path indicado en S3
            filesystem=fs,                  # Con la configuracion seleccionada
            coerce_timestamps="us",         # Pasando el timestamp a microsegundos (por spark)
            allow_truncated_timestamps=True # Para redondear los timestamp
        )

    print(f"Chunk #{chunk_number} processed")

C:\Users\Rodrigo\AppData\Local\Temp\ipykernel_9584\434628126.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  for chunk_number, chunk in enumerate(pd.read_csv(


Chunk #0 processed
Chunk #1 processed
Chunk #2 processed
Chunk #3 processed
Chunk #4 processed
Chunk #5 processed
Chunk #6 processed
Chunk #7 processed
Chunk #8 processed
Chunk #9 processed
Chunk #10 processed
Chunk #11 processed
Chunk #12 processed
Chunk #13 processed
Chunk #14 processed
Chunk #15 processed
Chunk #16 processed
Chunk #17 processed
Chunk #18 processed
Chunk #19 processed
Chunk #20 processed
Chunk #21 processed
Chunk #22 processed
Chunk #23 processed
Chunk #24 processed
Chunk #25 processed
Chunk #26 processed
Chunk #27 processed
Chunk #28 processed
Chunk #29 processed
Chunk #30 processed
Chunk #31 processed
Chunk #32 processed
Chunk #33 processed
Chunk #34 processed
Chunk #35 processed
Chunk #36 processed
Chunk #37 processed
Chunk #38 processed
Chunk #39 processed
Chunk #40 processed
Chunk #41 processed
Chunk #42 processed
Chunk #43 processed
Chunk #44 processed
Chunk #45 processed
Chunk #46 processed
Chunk #47 processed
Chunk #48 processed
Chunk #49 processed
Chunk #50 